# Notebook 13 – Preventing Data Leakage

## 1. What is Data Leakage?

Data leakage occurs when information from outside the training data is unintentionally used during model training.

This can happen when information from the test dataset, target variable, future data, or preprocessing steps is allowed to influence the training process.

### Why is Data Leakage Dangerous?

Data leakage can make a Machine Learning model appear to perform much better than it actually does on unseen data.

### AI/ML Relevance

Preventing data leakage is essential for obtaining reliable model evaluation and building models that generalize well to new data.

In [1]:
import pandas as pd

df = pd.read_csv("Titanic-Dataset.csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(list(df.columns))

Dataset Shape: (891, 12)

Columns:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


## 2. What is Target Leakage?

Target leakage occurs when a feature contains information that is directly or indirectly related to the target and would not be available at the time of prediction.

### Example

Suppose we want to predict whether a passenger survived.

If we create a feature using the actual `Survived` value, the model receives the answer directly.

### Why is it dangerous?

The model may achieve extremely high performance because it has access to information that would not be available in a real prediction situation.

### AI/ML Relevance

Target leakage can make a model appear highly accurate while making it unsuitable for real-world use.

In [2]:
# Incorrect example: using the target directly as a feature

X_wrong = df[["Pclass", "Age", "Fare", "Survived"]]
y = df["Survived"]

print("Features used in the incorrect workflow:")
print(list(X_wrong.columns))

print("\nWarning: 'Survived' is the target and should not be used as an input feature.")

Features used in the incorrect workflow:
['Pclass', 'Age', 'Fare', 'Survived']



## 3. What is Train-Test Contamination?

Train-test contamination occurs when information from the test dataset is allowed to influence the training process.

This can happen when we perform preprocessing, feature selection, or other data preparation on the complete dataset before splitting it.

### Example

Fitting a scaler on the complete dataset before train-test splitting allows information from the test data to influence the scaling parameters.

### AI/ML Relevance

Train-test contamination can produce overly optimistic evaluation results.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = df[["Age", "Fare"]].copy()
data = data.fillna(data.median())

# Incorrect workflow
scaler_wrong = StandardScaler()

data_scaled = scaler_wrong.fit_transform(data)

X_train_wrong, X_test_wrong = train_test_split(
    data_scaled,
    test_size=0.20,
    random_state=42
)

print("Incorrect workflow:")
print("Scaler was fitted before train-test splitting.")
print("This can cause train-test contamination.")

Incorrect workflow:
Scaler was fitted before train-test splitting.
This can cause train-test contamination.


## 4. What is Feature Leakage?

Feature leakage occurs when an input feature contains information that would only be known after the event we are trying to predict.

### Example

Suppose we want to predict whether a customer will cancel an order.

A feature such as `Cancellation_Reason` would be leakage if that information is recorded only after cancellation.

### Why is it dangerous?

The model learns information that would not exist when making the actual prediction.

### AI/ML Relevance

Features must represent information that would genuinely be available at prediction time.

In [4]:
# Example of a potentially leaking feature

example_data = pd.DataFrame({
    "Customer_Age": [25, 34, 42],
    "Order_Value": [500, 750, 1200],
    "Cancellation_Reason": ["Payment", "None", "Customer Request"]
})

print(example_data)

print("\n'Cancellation_Reason' could be a leaking feature")
print("if it becomes available only after the cancellation event.")

   Customer_Age  Order_Value Cancellation_Reason
0            25          500             Payment
1            34          750                None
2            42         1200    Customer Request

'Cancellation_Reason' could be a leaking feature
if it becomes available only after the cancellation event.


## 5. What is Preprocessing Leakage?

Preprocessing leakage occurs when preprocessing techniques learn information from data that should remain unseen.

Examples include:

- Calculating imputation values using the complete dataset.
- Fitting a scaler using training and test data together.
- Learning encoding information from the test dataset.
- Selecting features using the complete dataset.

### AI/ML Relevance

Preprocessing should generally be fitted using training data only and then applied to validation and test data.

In [5]:
from sklearn.impute import SimpleImputer

X = df[["Age", "Fare"]].copy()

X_train, X_test = train_test_split(
    X,
    test_size=0.20,
    random_state=42
)

# Correct approach
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

print("Imputer fitted only on training data.")
print("Test data was transformed using the training-fitted imputer.")

Imputer fitted only on training data.
Test data was transformed using the training-fitted imputer.


## 6. What is Temporal Leakage?

Temporal leakage occurs when information from the future is used to predict an event that happened earlier.

This is especially important for time-dependent datasets.

### Example

If we use 2025 information to predict an event that occurred in 2024, the model receives future information.

### Why is it dangerous?

The model can learn patterns that would not have been available when the prediction was actually made.

### AI/ML Relevance

Time-dependent problems should normally use earlier data for training and later data for validation or testing.

In [6]:
# Example temporal data

time_data = pd.DataFrame({
    "Year": [2021, 2022, 2023, 2024, 2025],
    "Sales": [100, 120, 140, 160, 180]
})

train_time = time_data[time_data["Year"] <= 2023]
test_time = time_data[time_data["Year"] > 2023]

print("Training Data:")
print(train_time)

print("\nTesting Data:")
print(test_time)

Training Data:
   Year  Sales
0  2021    100
1  2022    120
2  2023    140

Testing Data:
   Year  Sales
3  2024    160
4  2025    180


## 7. What are Examples of Data Leakage?

Common examples of data leakage include:

1. Using the target variable as an input feature.
2. Fitting a scaler before train-test splitting.
3. Calculating missing-value statistics using the complete dataset.
4. Selecting features using both training and test data.
5. Using future information to predict past events.
6. Using information that becomes available only after the target event.

### AI/ML Relevance

Identifying these situations helps ensure that model evaluation represents real-world performance.

In [7]:
print("Common Data Leakage Examples")
print("-----------------------------")
print("1. Target variable used as a feature")
print("2. Scaling before train-test split")
print("3. Imputation using complete dataset")
print("4. Feature selection using test data")
print("5. Future information used for past predictions")
print("6. Post-event information used as a feature")

Common Data Leakage Examples
-----------------------------
1. Target variable used as a feature
2. Scaling before train-test split
3. Imputation using complete dataset
4. Feature selection using test data
5. Future information used for past predictions
6. Post-event information used as a feature


## 8. How to Detect Data Leakage?

Data leakage can be detected by checking:

- Whether any feature directly contains the target.
- Whether features are created using future information.
- Whether preprocessing was fitted before splitting.
- Whether test data was used during feature selection.
- Whether validation or test performance is suspiciously high.
- Whether a feature would actually be available at prediction time.

### Important Question

For every feature, ask:

"Would this information be available at the exact time the prediction is made?"

If the answer is no, the feature may cause leakage.

### AI/ML Relevance

Leakage detection helps identify unrealistic features and preprocessing workflows before model deployment.

In [8]:
target = "Survived"

leakage_check = []

for column in df.columns:
    if column == target:
        leakage_check.append((column, "Target column"))
    else:
        leakage_check.append((column, "Review required"))

print("Leakage Review:")
for column, status in leakage_check:
    print(f"{column}: {status}")

Leakage Review:
PassengerId: Review required
Survived: Target column
Pclass: Review required
Name: Review required
Sex: Review required
Age: Review required
SibSp: Review required
Parch: Review required
Ticket: Review required
Fare: Review required
Cabin: Review required
Embarked: Review required


## 9. How to Prevent Data Leakage?

Data leakage can be prevented by following a proper Machine Learning workflow:

1. Separate features and target.
2. Split the dataset before fitting preprocessing techniques.
3. Fit imputers only on training data.
4. Fit scalers only on training data.
5. Fit encoders only on training data.
6. Perform feature selection using training data only.
7. Keep future information out of historical predictions.
8. Keep the test dataset completely separate until final evaluation.

### AI/ML Relevance

A leakage-free workflow provides a more realistic estimate of model performance on unseen data.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Separate features and target
X = df[["Pclass", "Age", "SibSp", "Parch", "Fare"]].copy()
y = df["Survived"]

# Split first
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Fit imputer only on training data
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Fit scaler only on training data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Correct leakage-prevention workflow completed.")
print("Training Shape:", X_train_scaled.shape)
print("Testing Shape:", X_test_scaled.shape)

Correct leakage-prevention workflow completed.
Training Shape: (712, 5)
Testing Shape: (179, 5)


## 10. Incorrect Workflow → Correct Workflow

### Incorrect Workflow

1. Load the complete dataset.
2. Perform preprocessing on the complete dataset.
3. Perform feature selection using the complete dataset.
4. Split the processed data into training and testing datasets.

### Problem

The preprocessing and feature selection steps may use information from the test dataset.

### Result

The test dataset can indirectly influence the training process.

### Impact

This may produce misleadingly high model performance.

---

### Correct Workflow

1. Load the dataset.
2. Separate features and target.
3. Split the data into training and testing datasets.
4. Fit preprocessing only on training data.
5. Transform training and testing data using the fitted preprocessing objects.
6. Train the model using training data.
7. Evaluate the final model on test data.

### Impact

This reduces the risk of data leakage and provides a more reliable evaluation.

In [10]:
# INCORRECT WORKFLOW

X = df[["Age", "Fare"]].copy()
X = X.fillna(X.median())

scaler = StandardScaler()

# Scaling before splitting
X_scaled = scaler.fit_transform(X)

X_train_wrong, X_test_wrong = train_test_split(
    X_scaled,
    test_size=0.20,
    random_state=42
)

print("INCORRECT WORKFLOW")
print("Preprocessing was fitted before splitting.")

INCORRECT WORKFLOW
Preprocessing was fitted before splitting.


In [11]:
# CORRECT WORKFLOW

X = df[["Age", "Fare"]].copy()

X_train, X_test = train_test_split(
    X,
    test_size=0.20,
    random_state=42
)

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

# Fit only on training data
X_train_imputed = imputer.fit_transform(X_train)
X_train_scaled = scaler.fit_transform(X_train_imputed)

# Transform test data
X_test_imputed = imputer.transform(X_test)
X_test_scaled = scaler.transform(X_test_imputed)

print("CORRECT WORKFLOW")
print("Preprocessing was fitted only on training data.")
print("Test data was transformed using the fitted preprocessing objects.")

CORRECT WORKFLOW
Preprocessing was fitted only on training data.
Test data was transformed using the fitted preprocessing objects.


# Required Documentation

## Problem

The Machine Learning workflow can produce misleading results if information from test data, target variables, future data, or post-event information enters the training process.

## Analysis

Data leakage occurs when information that should be unavailable during model training influences preprocessing, feature selection, feature engineering, or model training.

## Technique Selected

A leakage-prevention workflow was selected using train-test splitting before preprocessing and fitting preprocessing techniques only on training data.

## Reason

This method prevents information from the test dataset from influencing the learned preprocessing parameters and model development.

## Implementation

The target variable was separated from the input features. The dataset was split into training and testing datasets before preprocessing. Imputation and scaling were fitted only on the training data and then applied to the test data.

## Result

The preprocessing workflow no longer uses test data to learn imputation or scaling parameters. The test dataset remains separate for final evaluation.

## Impact

Preventing data leakage provides a more realistic estimate of Machine Learning model performance and improves confidence that the model can generalize to unseen data.

### Final Principle

**Never allow information that would not be available at prediction time to influence model training.**

In [12]:
print("Data Leakage Prevention Summary")
print("--------------------------------")
print("1. Target separated from features")
print("2. Data split before preprocessing")
print("3. Preprocessing fitted only on training data")
print("4. Test data transformed using fitted preprocessing")
print("5. Future information excluded from historical predictions")
print("6. Test data reserved for final evaluation")

Data Leakage Prevention Summary
--------------------------------
1. Target separated from features
2. Data split before preprocessing
3. Preprocessing fitted only on training data
4. Test data transformed using fitted preprocessing
5. Future information excluded from historical predictions
6. Test data reserved for final evaluation
